首先创建目录
mkdir -p ~/mcp_service
cd ~/mcp_service

创建 requirements.txt
cat > requirements.txt << 'EOF'
fastapi>=0.110.0
uvicorn[standard]>=0.29.0
python-jose[cryptography]>=3.3.0
python-dotenv>=1.0.0
EOF

进行操作openssl rand -hex 32
之后
cat > .env << EOF
JWT_SECRET=$JWT_SECRET
MCP_CLIENT_SECRET=MySuperSecretClientPassword123
JWT_EXPIRE_SECONDS=28800
EOF

In [ ]:
#写出config.py
import os
from dotenv import load_dotenv

load_dotenv()

JWT_SECRET = os.getenv("JWT_SECRET")
JWT_EXPIRE_SECONDS = int(os.getenv("JWT_EXPIRE_SECONDS", 28800))
CLIENT_SECRET = os.getenv("MCP_CLIENT_SECRET")
ALGORITHM = "HS256"

In [ ]:
#写出auth.py
from datetime import datetime, timedelta, timezone
from fastapi import HTTPException, status
from fastapi.security import HTTPBearer
from jose import JWTError, jwt
from config import JWT_SECRET, ALGORITHM, JWT_EXPIRE_SECONDS, CLIENT_SECRET

bearer_scheme = HTTPBearer(auto_error=False)

def create_token() -> str:
    now = datetime.now(timezone.utc)
    payload = {
        "iat": now,
        "exp": now + timedelta(seconds=JWT_EXPIRE_SECONDS),
        "sub": "mcp-client",
    }
    return jwt.encode(payload, JWT_SECRET, algorithm=ALGORITHM)

def require_auth(token: str):
    try:
        return jwt.decode(token, JWT_SECRET, algorithms=[ALGORITHM])
    except JWTError:
        raise HTTPException(
            status_code=status.HTTP_401_UNAUTHORIZED,
            detail="Invalid or expired token",
        )

def verify_client_secret(secret: str) -> bool:
    return secret == CLIENT_SECRET

In [ ]:
#写出sever.py
from fastapi import FastAPI, Request
from auth import create_token, require_auth, verify_client_secret

app = FastAPI()

@app.post("/mcp")
async def mcp(request: Request):
    try:
        body = await request.json()
    except Exception:
        return {
            "jsonrpc": "2.0",
            "id": None,
            "error": {"code": -32700, "message": "Parse error"},
        }

    method = body.get("method")
    params = body.get("params", {})
    req_id = body.get("id")

    if method == "tools/list":
        return {
            "jsonrpc": "2.0",
            "id": req_id,
            "result": {
                "tools": [
                    {
                        "name": "get_auth_token",
                        "description": "获取鉴权 Token（无需鉴权）",
                        "inputSchema": {
                            "type": "object",
                            "properties": {"client_secret": {"type": "string"}},
                            "required": ["client_secret"],
                        },
                    },
                    {
                        "name": "my_custom_action",
                        "description": "需要 Token 的自定义工具",
                        "inputSchema": {
                            "type": "object",
                            "properties": {"text": {"type": "string"}},
                            "required": ["text"],
                        },
                    },
                ]
            },
        }

    if method == "tools/call":
        name = params.get("name")
        args = params.get("arguments", {})

        if name == "get_auth_token":
            if not verify_client_secret(args.get("client_secret")):
                return {
                    "jsonrpc": "2.0",
                    "id": req_id,
                    "error": {"code": 403, "message": "Invalid client_secret"},
                }
            token = create_token()
            return {
                "jsonrpc": "2.0",
                "id": req_id,
                "result": {"content": [{"type": "text", "text": token}]},
            }

        if name == "my_custom_action":
            auth = request.headers.get("authorization", "")
            if not auth.startswith("Bearer "):
                return {
                    "jsonrpc": "2.0",
                    "id": req_id,
                    "error": {"code": 401, "message": "Missing token"},
                }
            try:
                require_auth(auth.split()[1])
            except Exception:
                return {
                    "jsonrpc": "2.0",
                    "id": req_id,
                    "error": {"code": 401, "message": "Invalid token"},
                }
            return {
                "jsonrpc": "2.0",
                "id": req_id,
                "result": {
                    "content": [
                        {"type": "text", "text": f"处理结果: {args.get('text', '').upper()}"}
                    ]
                },
            }

    return {
        "jsonrpc": "2.0",
        "id": req_id,
        "error": {"code": -32601, "message": "Method not found"},
    }

@app.get("/")
async def root():
    return {"status": "ok", "service": "mcp"}

@app.get("/health")
async def health():
    return {"status": "healthy"}

运行uvicorn server:app --uds /tmp/mcp.sock